### Home Credit Default Risk - Dataset Audit
Goal:
1. Load all available tables
2. Check shapes and columns
3. Understand key identifiers
4. Map relationships between tables
5. Decide how feature engineering should be done later

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append("..")

from src.config import RAW_DATA_DIR, SAMPLE_DATA_DIR

In [3]:
USE_SAMPLE = False
DATA_DIR = SAMPLE_DATA_DIR if USE_SAMPLE else RAW_DATA_DIR

DATA_DIR

WindowsPath('C:/Coding/Home-Credit-Default-Risk/data/raw')

In [4]:
file_names = {
    "application_train": "application_train.csv",
    "application_test": "application_test.csv",
    "bureau": "bureau.csv",
    "bureau_balance": "bureau_balance.csv",
    "previous_application": "previous_application.csv",
    "installments_payments": "installments_payments.csv",
    "POS_CASH_balance": "POS_CASH_balance.csv",
    "credit_card_balance": "credit_card_balance.csv",
    "columns_description": "HomeCredit_columns_description.csv",
}

In [7]:
tables = {}

for name, file_name in file_names.items():
    path = DATA_DIR / file_name
    try:
        if name == "columns_description":
            tables[name] = pd.read_csv(path, encoding="cp1252")
        else:
            tables[name] = pd.read_csv(path)
        print(f"{name}: loaded successfully - shape = {tables[name].shape}")
    except Exception as e:
        print(f"{name}: failed to load - {e}")

application_train: loaded successfully - shape = (307511, 122)
application_test: loaded successfully - shape = (48744, 121)
bureau: loaded successfully - shape = (1716428, 17)
bureau_balance: loaded successfully - shape = (27299925, 3)
previous_application: loaded successfully - shape = (1670214, 37)
installments_payments: loaded successfully - shape = (13605401, 8)
POS_CASH_balance: loaded successfully - shape = (10001358, 8)
credit_card_balance: loaded successfully - shape = (3840312, 23)
columns_description: loaded successfully - shape = (219, 5)


In [8]:
summary = []

for name, df in tables.items():
    summary.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum())
    })

summary_df = pd.DataFrame(summary).sort_values("rows", ascending=False)
summary_df

,table,rows,columns,missing_cells
3,bureau_balance,27299925,3,0
5,installments_payments,13605401,8,5810
6,POS_CASH_balance,10001358,8,52158
7,credit_card_balance,3840312,23,5877356
2,bureau,1716428,17,3939947
4,previous_application,1670214,37,11109336
0,application_train,307511,122,9152465
1,application_test,48744,121,1404419
8,columns_description,219,5,133


In [9]:
for name, df in tables.items():
    print("=" * 80)
    print(name)
    display(df.head())

application_train


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


application_test


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


bureau


...

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


credit_card_balance


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


columns_description


,Unnamed: 0,Table,Row,Description,Special
0,1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
1,2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
2,5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
3,6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
4,7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN


In [10]:
key_cols = ["SK_ID_CURR", "SK_ID_PREV", "SK_ID_BUREAU"]

key_summary = []

for name, df in tables.items():
    row = {"table": name}
    for col in key_cols:
        if col in df.columns:
            row[f"{col}_exists"] = True
            row[f"{col}_nunique"] = df[col].nunique()
        else:
            row[f"{col}_exists"] = False
            row[f"{col}_nunique"] = np.nan
    key_summary.append(row)

key_summary_df = pd.DataFrame(key_summary)
key_summary_df

,table,SK_ID_CURR_exists,SK_ID_CURR_nunique,SK_ID_PREV_exists,SK_ID_PREV_nunique,SK_ID_BUREAU_exists,SK_ID_BUREAU_nunique
0,application_train,True,307511.0,False,NaN,False,NaN
1,application_test,True,48744.0,False,NaN,False,NaN
2,bureau,True,305811.0,False,NaN,True,1716428.0
3,bureau_balance,False,NaN,False,NaN,True,817395.0
4,previous_application,True,338857.0,True,1670214.0,False,NaN
5,installments_payments,True,339587.0,True,997752.0,False,NaN
6,POS_CASH_balance,True,337252.0,True,936325.0,False,NaN
7,credit_card_balance,True,103558.0,True,104307.0,False,NaN
8,columns_description,False,NaN,False,NaN,False,NaN


In [11]:
train_df = tables["application_train"]

print("Rows:", len(train_df))
print("Unique SK_ID_CURR:", train_df["SK_ID_CURR"].nunique())
print("Duplicate SK_ID_CURR:", train_df["SK_ID_CURR"].duplicated().sum())

Rows: 307511
Unique SK_ID_CURR: 307511
Duplicate SK_ID_CURR: 0


In [18]:
train_ids = set(tables["application_train"]["SK_ID_CURR"].dropna().unique())
test_ids = set(tables["application_test"]["SK_ID_CURR"].dropna().unique())
all_app_ids = train_ids | test_ids

coverage_rows = []

for table_name in ["bureau", "previous_application", "installments_payments", "POS_CASH_balance", "credit_card_balance"]:
    df = tables[table_name]
    if "SK_ID_CURR" in df.columns:
        table_ids = set(df["SK_ID_CURR"].dropna().unique())
        coverage_rows.append({
            "table": table_name,
            "train_coverage": len(train_ids & table_ids) / len(train_ids),
            "test_coverage": len(test_ids & table_ids) / len(test_ids),
            "all_application_coverage": len(all_app_ids & table_ids) / len(all_app_ids),
        })

pd.DataFrame(coverage_rows)

,table,train_coverage,test_coverage,all_application_coverage
0,bureau,0.856851,0.868209,0.858405
1,previous_application,0.946493,0.980634,0.951164
2,installments_payments,0.948399,0.983588,0.953213
3,POS_CASH_balance,0.941248,0.980798,0.946659
4,credit_card_balance,0.282608,0.341642,0.290685


In [13]:
bureau_df = tables["bureau"]
bureau_balance_df = tables["bureau_balance"]

print("bureau SK_ID_BUREAU unique:", bureau_df["SK_ID_BUREAU"].nunique())
print("bureau_balance SK_ID_BUREAU unique:", bureau_balance_df["SK_ID_BUREAU"].nunique())

common_bureau_ids = set(bureau_df["SK_ID_BUREAU"]).intersection(set(bureau_balance_df["SK_ID_BUREAU"]))
print("Common SK_ID_BUREAU count:", len(common_bureau_ids))

bureau SK_ID_BUREAU unique: 1716428
bureau_balance SK_ID_BUREAU unique: 817395
Common SK_ID_BUREAU count: 774354


In [14]:
for name in ["previous_application", "installments_payments", "POS_CASH_balance", "credit_card_balance"]:
    df = tables[name]
    print(f"{name}:")
    print("  SK_ID_CURR unique:", df["SK_ID_CURR"].nunique() if "SK_ID_CURR" in df.columns else "N/A")
    print("  SK_ID_PREV unique:", df["SK_ID_PREV"].nunique() if "SK_ID_PREV" in df.columns else "N/A")
    print("  rows:", len(df))
    print()

previous_application:
  SK_ID_CURR unique: 338857
  SK_ID_PREV unique: 1670214
  rows: 1670214

installments_payments:
  SK_ID_CURR unique: 339587
  SK_ID_PREV unique: 997752
  rows: 13605401

POS_CASH_balance:
  SK_ID_CURR unique: 337252
  SK_ID_PREV unique: 936325
  rows: 10001358

credit_card_balance:
  SK_ID_CURR unique: 103558
  SK_ID_PREV unique: 104307
  rows: 3840312



In [15]:
table_roles = pd.DataFrame([
    {"table": "application_train", "role": "main training table", "merge_level": "customer/application level"},
    {"table": "application_test", "role": "main test table", "merge_level": "customer/application level"},
    {"table": "bureau", "role": "external credit history", "merge_level": "aggregate to SK_ID_CURR"},
    {"table": "bureau_balance", "role": "monthly bureau history", "merge_level": "aggregate to SK_ID_BUREAU then SK_ID_CURR"},
    {"table": "previous_application", "role": "past Home Credit applications", "merge_level": "aggregate to SK_ID_CURR"},
    {"table": "installments_payments", "role": "payment history", "merge_level": "aggregate to SK_ID_PREV/SK_ID_CURR"},
    {"table": "POS_CASH_balance", "role": "monthly POS/cash history", "merge_level": "aggregate to SK_ID_PREV/SK_ID_CURR"},
    {"table": "credit_card_balance", "role": "monthly credit card history", "merge_level": "aggregate to SK_ID_PREV/SK_ID_CURR"},
])

table_roles

,table,role,merge_level
0,application_train,main training table,customer/application level
1,application_test,main test table,customer/application level
2,bureau,external credit history,aggregate to SK_ID_CURR
3,bureau_balance,monthly bureau history,aggregate to SK_ID_BUREAU then SK_ID_CURR
4,previous_application,past Home Credit applications,aggregate to SK_ID_CURR
5,installments_payments,payment history,aggregate to SK_ID_PREV/SK_ID_CURR
6,POS_CASH_balance,monthly POS/cash history,aggregate to SK_ID_PREV/SK_ID_CURR
7,credit_card_balance,monthly credit card history,aggregate to SK_ID_PREV/SK_ID_CURR


In [19]:
relationship_diagram = """
application_train / application_test
    └── SK_ID_CURR
        ├── bureau
        │     └── SK_ID_BUREAU
        │           └── bureau_balance
        ├── previous_application
        │     └── SK_ID_PREV
        │           ├── installments_payments
        │           ├── POS_CASH_balance
        │           └── credit_card_balance
        └── current_application
              └── SK_ID_CURR
                    ├── static application features
                    ├── external credit history
                    │     └── monthly bureau history
                    ├── previous Home Credit applications
                    ├── installment repayment history
                    ├── POS/cash monthly history
                    └── credit card monthly history
"""

print(relationship_diagram)


application_train / application_test
    └── SK_ID_CURR
        ├── bureau
        │     └── SK_ID_BUREAU
        │           └── bureau_balance
        ├── previous_application
        │     └── SK_ID_PREV
        │           ├── installments_payments
        │           ├── POS_CASH_balance
        │           └── credit_card_balance
        └── current_application
              └── SK_ID_CURR
                    ├── static application features
                    ├── external credit history
                    │     └── monthly bureau history
                    ├── previous Home Credit applications
                    ├── installment repayment history
                    ├── POS/cash monthly history
                    └── credit card monthly history



Day 1 conclusions

- `application_train` is the main supervised training table.
- The dataset is relational and contains several one-to-many historical tables.
- Auxiliary tables should be aggregated before merging to the main table.
- The next step is EDA on `application_train` and a baseline model using only the main table.